# 📖 Theory Notebook — Tutorial 3: EEG Analysis & Classification
### 4th Year · Biomedical Signals & Applications

---

> **How to use this notebook:**
> Open it side-by-side with `Tutorial_3_EEG_Analysis_and_Classification.ipynb`.
> Before running each analysis step, read the matching theory section here.
> No signal processing background required.

---

## What is this tutorial about?

In Tutorial 2 we cleaned the raw EEG.
Now we ask the interesting question: **what is the brain actually doing during motor imagery?**

We will approach this from four angles, each revealing something different:

| Analysis | Question it answers | What you see |
|----------|-------------------|--------------|
| **ERP** | Does average brain voltage differ over time? | A waveform: voltage vs time |
| **Power spectrum** | Are some frequencies stronger in one condition? | A graph: power vs frequency |
| **Time-frequency map** | Which frequencies change, and when? | A 2D colour image |
| **Classification** | Can a computer predict which hand was imagined? | An accuracy percentage |

---

## What is motor imagery?

**Motor imagery** means imagining a movement without actually doing it.

When you imagine moving your right hand, the motor cortex in your brain activates — even though your hand doesn't move. This brain activity produces measurable changes in the EEG.

This is the foundation of **brain-computer interfaces (BCIs)** for people with paralysis: they imagine a movement, the computer reads the brain signal, and executes a command.

In this tutorial, subjects imagined moving either their **left hand** or their **right hand** in response to a cue. Our job is to find the brain signal difference between these two conditions — and then teach a classifier to recognise it.

---

---
## 1. The Most Important Fact: The Contralateral Rule

Before we look at any analysis, you need to know this one anatomical fact — it explains everything that follows.

---

### Your brain controls the opposite side of your body

The motor cortex (the brain region that controls movement) is organised **contralaterally**:

- The **left hemisphere** controls the **right hand**
- The **right hemisphere** controls the **left hand**

This is called contralateral control, and it applies to actual movement **and** to imagined movement.

---

### How this maps to EEG electrodes

In the standard 10-20 electrode system, the electrodes sit directly over the brain regions they record from:

| Electrode | Brain region underneath | Most active during |
|-----------|------------------------|-------------------|
| **C3** | Left motor cortex | Right hand movement or imagery |
| **Cz** | Central motor area (midline) | Bilateral or leg activity |
| **C4** | Right motor cortex | Left hand movement or imagery |

---

> 🟢 **Analogy:** Think of it like crossed wires. The left side of the brain runs the right side of the body, and vice versa. C3 is on the left of the head — it sits over the left hemisphere — so it becomes most active when you use or imagine using your RIGHT hand.

---

### Why this matters for classification

When a subject imagines moving their **right** hand:
→ The left motor cortex activates
→ **C3 shows stronger brain activity**
→ C4 shows less activity

When a subject imagines moving their **left** hand:
→ The right motor cortex activates
→ **C4 shows stronger brain activity**
→ C3 shows less activity

**This left-right difference between C3 and C4 is exactly what our classifier will learn to recognise.** Everything in this tutorial is about measuring this difference in different ways.

---

---
## 2. Event-Related Potentials (ERPs) — Averaging to Find the Signal

---

### The problem: brain signals are hidden in noise

If you look at the EEG from a single trial, you mostly see the brain's ongoing background activity — spontaneous oscillations that have nothing to do with the specific event that just occurred. The brain's actual response to the event is much smaller and completely invisible in a single trial.

---

### The solution: averaging many trials

Here is the key insight:
- The brain's **response** to a repeated event is roughly **the same every time** — it is locked in time to the event
- The background **noise** is **random** — it has no relationship to when the event occurred

When we average many trials together:
- The consistent response **adds up** (same signal + same signal = twice as strong)
- The random noise **cancels out** (positive and negative random fluctuations cancel each other)

After averaging many trials, the brain's response emerges clearly from the noise.

---

> 🟢 **Analogy:** Imagine trying to photograph a faint star through a cloudy sky. Any single 1-second photo shows mostly clouds (noise). But if you take 100 photos of the same patch of sky and average the pixels, the random cloud patterns cancel and the star becomes visible.
>
> ERP averaging does exactly this — the "star" is the brain's response to the stimulus.

---

### What an ERP waveform looks like

An ERP is plotted as **voltage vs time**, with time 0 being the moment the cue appeared.
- Positive voltage peaks are labelled **P** (e.g. P300 = positive peak around 300 ms)
- Negative voltage peaks are labelled **N** (e.g. N200 = negative peak around 200 ms)

For **motor imagery**, we look at:
- Slow voltage changes at C3 and C4 during the 0–3 second imagery window
- The **difference** between the two channels — right-hand imagery should make C3 more active than C4, and vice versa

---

### What to look for in the ERP plot

When you compare the ERP at C3 for left-hand vs right-hand imagery:
- The two curves should **diverge** after the cue (t = 0)
- Right-hand imagery should produce a **stronger or more negative** deflection at C3 (because C3 is contralateral to the right hand)
- The same comparison at C4 should show the **opposite pattern**

---

---
## 3. Frequency Analysis — What Oscillations Is the Brain Using?

---

### The brain oscillates

Neurons in the brain fire in rhythmic pulses, and large groups of neurons can synchronise into waves. Different mental states produce different dominant oscillation speeds (frequencies).

You can think of EEG as listening to a complex instrument. The brain plays different "notes" (frequencies) depending on what it is doing.

---

### EEG frequency bands

| Band | Speed | What it means |
|------|-------|---------------|
| **Delta (0.5–4 Hz)** | Very slow | Deep sleep; the brain is nearly off |
| **Theta (4–8 Hz)** | Slow | Memory encoding; drowsiness; daydreaming |
| **Alpha (8–13 Hz)** | Medium | Eyes-closed relaxation; the brain's "idle" state |
| **Mu (8–12 Hz)** | Medium | Same speed as alpha, but specifically over the **motor cortex** — the motor cortex's idle rhythm |
| **Beta (13–30 Hz)** | Fast | Active thinking; movement; motor imagery |
| **Gamma (30+ Hz)** | Very fast | Focused attention; complex cognition |

---

### The mu rhythm — the key to motor imagery

The **mu rhythm** is the motor cortex's version of alpha. When the motor cortex is at rest (not planning any movement), it produces a rhythmic oscillation at 8–12 Hz — the mu rhythm.

When you move your hand — or even just **imagine** moving it — the mu rhythm **disappears**.

This disappearance is called **ERD** (Event-Related Desynchronisation) and it is the core signal we will measure.

---

> 🟢 **Analogy:** The mu rhythm is like a car engine idling in neutral.
> When the driver decides to move — even just *thinks* about moving — the engine engages and the rhythmic idle stops.
> When the car parks again, the idle resumes.
>
> Mu ERD = the idle rhythm disappearing because the motor cortex is "doing something"

---

### Band power

**Band power** is a single number that summarises how much energy the EEG contains in a specific frequency range.

For motor imagery: we measure alpha/mu band power (8–13 Hz) at C3 and C4.

- **High mu band power** at C3 → left motor cortex is idle → probably left-hand condition
- **Low mu band power** at C3 → left motor cortex is active → probably right-hand condition (ERD is occurring)

This asymmetry in band power between C3 and C4 is what our features will capture.

---

---
## 4. Time-Frequency Analysis — When Does Each Frequency Change?

---

### The limitation of a simple spectrum

When we compute the power spectrum of a 3-second epoch, we get a single graph averaged over those 3 seconds. We cannot see whether the alpha power was high at the beginning and low at the end, or exactly when the ERD started.

A **time-frequency representation (TFR)** solves this. It is a 2D image:
- **Horizontal axis (x):** Time in seconds
- **Vertical axis (y):** Frequency in Hz
- **Colour:** How much power exists at that frequency at that moment

---

> 🟢 **Analogy:** A power spectrum is like a recipe that lists all the ingredients in a dish — flour, sugar, butter. Useful, but it tells you nothing about when each ingredient was added or how the mixture changed.
>
> A time-frequency map is like a video of the cooking process — you can see every ingredient appear and change in real time.

---

### What to expect in a motor imagery TFR

For **right-hand imagery** at electrode C3:

```
Time →         -0.5s  |  0s (cue)  |  1s   |  2s   |  3s   |  4s
                       |            |       |       |       |
High freq (26Hz)  ─────┤────────────┤───────┤───────┤───────┤─────
                       |            | BLUE  | BLUE  | BLUE  |
Alpha/Mu (10Hz)   ─────┤────────────┤─ ERD ─┤─ ERD ─┤─ ERD ─┤─────
                       |            |       |       |       | RED
Beta (20Hz)       ─────┤────────────┤───────┤───────┤───────┤(rebound)
Low freq (6Hz)    ─────┤────────────┤───────┤───────┤───────┤─────
                 Baseline           ←── Motor imagery ──→  Rest
```

- **Blue** = power **below** baseline = ERD (desynchronisation = the mu rhythm disappearing)
- **Red** = power **above** baseline = ERS (synchronisation = the mu rhythm returning, sometimes overshooting)

---

### ERD and ERS explained

**ERD — Event-Related Desynchronisation**
The motor cortex is active. Neurons are firing but NOT in a synchronised rhythmic pattern — they are busy doing work. The 8–12 Hz mu rhythm breaks down and disappears. Power drops below baseline → **blue** in the TFR.

**ERS — Event-Related Synchronisation**
The motor cortex returns to its idle state after the imagery ends. The neurons re-synchronise. Power rises back above baseline → **red** in the TFR. This often "overshoots" — called the **beta rebound** — before settling back to normal.

---

### Morlet wavelets — the tool for computing TFRs

A **Morlet wavelet** is a short oscillation at a specific frequency, shaped like a wave packet that fades to zero at its edges.

To find how much 10 Hz activity exists at time t = 1.5 s:
1. Take a 10 Hz Morlet wavelet
2. Slide it along the EEG signal
3. Measure how well it matches the signal at each moment

Repeat this for every frequency from 6 to 40 Hz and every time point.
The result is the complete TFR.

The parameter **n_cycles** controls a trade-off:
- **Fewer cycles** → better at telling you *when* something happens, but less precise about *which frequency*
- **More cycles** → better at identifying the exact frequency, but smears out *when* it happens

For motor imagery, `n_cycles = frequency / 2` gives a good balance.

---

---
## 5. The Motor Cortex Story — What Really Happens During Imagery

Let's put it all together into a clear timeline.

---

### What happens in the brain during right-hand motor imagery

| Time | Event | What you see in EEG |
|------|-------|---------------------|
| −0.5 to 0 s | Subject at rest; motor cortex idling | Stable mu rhythm at C3 and C4 — they look equal |
| 0 s | The right-hand cue appears on screen | Visual cortex activates briefly (not visible at C3/C4) |
| 0 to 0.5 s | Subject begins imagining right-hand movement | Mu power starts to drop at C3 (left motor cortex activating) |
| 0.5 to 2.5 s | Active motor imagery | Strong mu and beta ERD at C3; C4 stays relatively quiet or shows mild increase |
| ~3 s | Imagery ends (rest cue) | Motor cortex returns to idle — mu and beta power rebounds above baseline (beta rebound) |

---

### The lateralisation — the signal that makes classification possible

At t = 1.5 s during the imagery:
- **C3** shows strong ERD (left motor cortex is working = imagining right hand)
- **C4** shows little or no ERD (right motor cortex is idle)

For **left-hand imagery**, the pattern is exactly reversed:
- C4 shows strong ERD
- C3 shows little ERD

**This difference between C3 and C4 is the signal our classifier will use.**

---

> 💡 **Key insight:**
> We don't need to understand the exact neural mechanism.
> We just need to measure whether C3 or C4 has lower mu/beta power.
> Whichever is lower tells us which hand was imagined.
> This is the entire basis of motor imagery BCI.

---

---
## 6. Feature Extraction — Turning EEG into Numbers for a Classifier

---

### The problem: too much data

Each clean epoch is a 2D array: **64 channels × 561 time points = 35,904 numbers**.

A machine learning classifier cannot work with this raw 2D array. It needs a single row of numbers per trial — a **feature vector**.

Feature extraction is the step where we transform each epoch from 35,904 numbers into a small set of meaningful measurements.

---

> 🟢 **Analogy:** Imagine summarising a 2-hour football match with statistics. Instead of watching all 120 minutes, you describe it with: goals scored, possession percentage, shots on target, fouls committed, corners. These 5 numbers capture what matters about the match. You can now compare 80 matches easily.
>
> EEG features do the same — we extract the statistics of each trial that are most useful for distinguishing left-hand from right-hand imagery.

---

### What features do we extract?

For motor imagery, the most informative features are:

**Alpha/mu band power (8–13 Hz)** and **Beta band power (13–30 Hz)** at the motor channels:
- C3, Cz, C4, FC3, FC4

This gives us **10 features per trial**: 5 channels × 2 frequency bands.

| Feature | What it measures |
|---------|-----------------|
| Alpha_C3 | Left motor cortex mu power — low during right-hand imagery (ERD) |
| Alpha_C4 | Right motor cortex mu power — low during left-hand imagery (ERD) |
| Beta_C3 | Left motor cortex beta power — also drops during right-hand imagery |
| Beta_C4 | Right motor cortex beta power — also drops during left-hand imagery |
| ... | (same logic for Cz, FC3, FC4) |

---

### The feature matrix

After computing features for all trials, we build a table:
- One **row** per trial
- One **column** per feature

For 80 trials and 10 features: the feature matrix has shape **80 × 10**.

This table is what the classifier sees. Our goal: trials for left-hand imagery should cluster in one region of this table, and right-hand trials in another.

---

### Why log-transform?

Band power values have a skewed distribution — most trials have moderate power, but a few have unusually high power (residual artefacts or spontaneous alpha bursts). This skewedness can confuse classifiers.

Taking the **logarithm** of each band power value compresses the large values and makes the distribution more symmetric and bell-shaped. This makes the classifier work better.

---

---
## 7. Classification — Teaching the Computer to Distinguish Left from Right

---

### What is a classifier?

A classifier is an algorithm that:
1. **Learns** from labelled examples (training data)
2. **Predicts** the label of new, unseen examples (test data)

In our case:
- **Training:** Show the classifier 64 trials with their correct labels (left or right hand)
- **Testing:** Show it 16 new trials and ask it to predict left or right

---

### LDA — Linear Discriminant Analysis

**LDA** is the most widely used classifier for motor imagery BCI. It is simple, fast, and surprisingly effective.

The idea: plot all left-hand trials as blue dots and all right-hand trials as red dots in feature space. LDA finds the **straight line** that divides the blue cloud from the red cloud as cleanly as possible.

The "best" line is the one that:
- Maximises the distance between the two group centres (classes are far apart)
- Minimises the spread within each group (each class is tightly clustered)

When a new trial arrives, you compute its features, plot it as a dot, and see which side of the line it falls on. That is the prediction.

---

> 🟢 **Analogy:** Two piles of gold coins and silver coins mixed on a table. LDA finds the line on the table that puts as many gold coins on one side and silver coins on the other as possible. Any new coin goes to the side of the line it falls on.

---

### Cross-validation — testing honestly

You cannot evaluate how well a classifier works by testing it on the same data it was trained on. It will score perfectly — because it memorised the training examples rather than learning a general pattern.

**5-fold cross-validation** solves this:

```
All 80 trials, split into 5 equal groups of 16:

Round 1: Train on groups 2,3,4,5 → Test on group 1
Round 2: Train on groups 1,3,4,5 → Test on group 2
Round 3: Train on groups 1,2,4,5 → Test on group 3
Round 4: Train on groups 1,2,3,5 → Test on group 4
Round 5: Train on groups 1,2,3,4 → Test on group 5

Final accuracy = average of the 5 test accuracies
```

Every trial is used as a test exactly once. The result is an **honest estimate** of how the classifier will perform on truly new data.

---

### What accuracy to expect

| Accuracy | What it means |
|----------|--------------|
| **50%** | Chance level — the classifier is guessing randomly |
| **55–65%** | Weak — some signal, but features or classifier need improvement |
| **65–75%** | Typical for band power + LDA in motor imagery |
| **75–85%** | Good — the subject shows clear, lateralised ERD |
| **85%+** | Excellent — very clean data or advanced methods |

Even 65–70% accuracy can be clinically useful: over many repeated trials, the correct command wins by majority vote with high confidence.

---

---
## 8. CSP — Smarter Spatial Filtering

---

### The problem with choosing channels by hand

In the feature extraction step, we manually chose which channels to use (C3, Cz, C4, FC3, FC4) based on our neuroscientific knowledge. But:

- Every person's brain is slightly different — the motor cortex might be shifted a few centimetres
- The best signal might come from a combination of nearby electrodes, not a single one
- Choosing channels by hand could miss the most informative signal for a specific individual

**CSP (Common Spatial Patterns)** finds the optimal combination of all 64 channels automatically from the data — without requiring any prior knowledge.

---

> 🟢 **Analogy:** Instead of you choosing which single microphone to use in a noisy room, CSP asks: "What is the perfect *blend* of all microphones that maximally highlights the difference between the two conditions?" Then it creates that optimal blend automatically.

---

### What CSP does

CSP creates **virtual channels** — each one is a weighted sum of all 64 real channels.

It optimises the weights so that:
- Virtual channel 1 **maximises** variance for left-hand condition while **minimising** it for right-hand
- Virtual channel 2 does the opposite

The log-variance of each virtual channel becomes a feature. Because the virtual channels are specifically designed to separate the two conditions, these features are much more discriminating than hand-picked band power values.

---

### What CSP spatial patterns look like

After running CSP, you can visualise the spatial patterns as scalp maps:
- **Pattern 1–2** (optimised for one condition): should show focal activity near **C4**
- **Pattern 3–4** (optimised for the other condition): should show focal activity near **C3**

This matches the neuroscience — the patterns should highlight the motor cortex of each hemisphere.

---

### CSP + LDA: the gold standard for motor imagery BCI

The combination of CSP for feature extraction and LDA for classification has been the dominant approach in motor imagery BCI for over 20 years. It consistently outperforms hand-crafted features while remaining interpretable and fast.

| Method | Typical accuracy | Notes |
|--------|-----------------|-------|
| Band power + LDA | 65–75% | Simple; interpretable; hand-designed features |
| CSP + LDA | 70–80% | Data-driven spatial filter; better for individual differences |
| Deep learning (e.g. EEGNet) | 75–85% | Learns from raw data; needs more trials to train |

---

---
## 9. Why Does This Matter? — Brain-Computer Interfaces

---

### Who needs BCIs?

Everything we built in Tutorial 3 — ERD, band power, CSP, LDA — comes together in **brain-computer interfaces (BCIs)**: systems that translate brain signals into computer commands without any muscle movement.

BCIs are designed for people with severe motor disabilities:

**ALS (Lou Gehrig's disease):**
Patients progressively lose all voluntary muscle control. In the final stages they cannot move, speak, or type. A BCI lets them communicate and control devices using only their brain activity.

**Spinal cord injury:**
The brain is intact and working, but the signal cannot travel down the spinal cord to reach the muscles. A BCI can bridge this gap — reading the brain's motor intention and directly stimulating the muscles or controlling a robotic arm.

**Stroke rehabilitation:**
After a stroke, patients often lose motor function. Research shows that if a patient performs motor imagery while receiving visual feedback that their imagined movement "succeeded," the brain gradually rewires itself and recovers function. This is called neuroplasticity-based rehabilitation.

---

### The closed feedback loop

A motor imagery BCI works in real time:

```
1. User imagines moving left or right hand
2. EEG records brain activity
3. Preprocessing (filter + artefact removal)  ← Tutorial 2
4. Feature extraction (band power or CSP)     ← Tutorial 3
5. LDA predicts: left or right?
6. Computer executes the command (cursor moves)
7. User sees the result on screen
8. Brain learns to produce a clearer ERD signal  ← neuroplasticity!
```

Step 8 — the feedback — is what makes this a **closed loop**. The user is not just a passive signal source. When they see the cursor move in the right direction, their brain learns to produce a stronger ERD. Accuracy typically improves significantly over several training sessions.

---

### Performance in real clinical use

| Scenario | Typical accuracy |
|----------|-----------------|
| Naive user, first session | 55–65% |
| After several training sessions | 70–85% |
| Expert BCI user | 90%+ |
| Clinically useful threshold | ~70% (with majority vote over multiple trials) |

---

---
## 10. Quick Glossary

| Term | Plain-English meaning |
|------|-----------------------|
| **ERP** | Event-Related Potential — average brain response time-locked to an event |
| **P300** | A positive voltage peak ~300 ms after a rare or unexpected stimulus |
| **Contralateral** | On the opposite side — left hemisphere controls right hand |
| **Power spectrum** | A graph showing how much energy the EEG contains at each frequency |
| **Band power** | Average power within a frequency band — one number per (channel, band) |
| **Mu rhythm** | The 8–12 Hz idle oscillation of the motor cortex |
| **ERD** | Event-Related Desynchronisation — power DECREASES (mu rhythm disappears during imagery) |
| **ERS** | Event-Related Synchronisation — power INCREASES (motor cortex returns to idle) |
| **Beta rebound** | ERS at 18–26 Hz after movement ends — motor cortex overshooting back to idle |
| **TFR** | Time-Frequency Representation — a 2D colour map of power vs time and frequency |
| **Morlet wavelet** | A short oscillation used to detect a specific frequency at a specific time |
| **n_cycles** | Controls the time vs frequency precision trade-off in wavelet analysis |
| **Feature vector** | A small set of numbers summarising one EEG epoch for use in a classifier |
| **Feature matrix** | The table of feature vectors — one row per trial, one column per feature |
| **LDA** | Linear Discriminant Analysis — finds the straight line best separating two classes |
| **Cross-validation** | Method for honest accuracy estimation — test only on data not used in training |
| **CSP** | Common Spatial Patterns — finds the optimal combination of channels to separate conditions |
| **BCI** | Brain-Computer Interface — translates brain signals into computer commands |
| **Lateralisation** | The fact that C3 and C4 show opposite patterns for left vs right hand imagery |

---

> ✅ **You are now ready to run Tutorial 3.**
>
> The single most important thing to watch for in every analysis:
> **Does C3 show more ERD for right-hand imagery, and C4 more ERD for left-hand imagery?**
>
> If yes — the contralateral signal is present, and classification will work.
> If no — check the preprocessing, or this subject may simply not produce a strong ERD.